# 01 Dataset Inspection and Cleaning

Load the CICMalDroid 2020 CSV files, build a stable ordered master dataframe, immediately split it into two contiguous raw shards, and then inspect/clean only the assigned shard. No balancing, poisoning, or training is done here.


This cell imports the core libraries and defines the shard assignment, deletion behavior, and project paths used by the rest of the notebook.

In [ ]:
from pathlib import Path
import re
from functools import reduce

import pandas as pd
from IPython.display import display

# Each machine edits this value and keeps only its assigned intermediate raw shard.
ASSIGNED_SHARD_ID = 1  # 1 or 2
DELETE_UNASSIGNED_SHARD = True
NUM_SHARDS = 2

# Make paths work from either the project root or the notebooks/ folder.
PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

RAW_SHARD_PATHS = {
    1: INTERIM_DIR / "raw_shard1.csv",
    2: INTERIM_DIR / "raw_shard2.csv",
}

SOURCE_DATASET = "CICMalDroid2020"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

if ASSIGNED_SHARD_ID not in RAW_SHARD_PATHS:
    raise ValueError("ASSIGNED_SHARD_ID must be 1 or 2.")

ASSIGNED_RAW_SHARD_PATH = RAW_SHARD_PATHS[ASSIGNED_SHARD_ID]
CLEANED_PATH = INTERIM_DIR / f"cleaned_cicmaldroid_shard{ASSIGNED_SHARD_ID}.csv"

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DIR)
print("Assigned shard:", ASSIGNED_SHARD_ID)
print("Delete unassigned shard:", DELETE_UNASSIGNED_SHARD)
print("Assigned raw shard:", ASSIGNED_RAW_SHARD_PATH)
print("Assigned cleaned output:", CLEANED_PATH)


This cell discovers the raw CICMalDroid files, lists them for inspection, and selects CSV files for loading.

In [ ]:
# Raw file discovery is allowed before splitting. Detailed inspection happens after the split.
raw_files = sorted(path for path in RAW_DIR.rglob("*") if path.is_file())
if not raw_files:
    raise FileNotFoundError(f"No files found in {RAW_DIR}. Place the CICMalDroid 2020 files there first.")

file_inventory = pd.DataFrame([
    {
        "path": str(path.relative_to(PROJECT_ROOT)),
        "suffix": path.suffix.lower(),
        "size_mb": round(path.stat().st_size / (1024 * 1024), 3),
    }
    for path in raw_files
])
display(file_inventory)

csv_paths = sorted(path for path in raw_files if path.suffix.lower() == ".csv")
if not csv_paths:
    raise FileNotFoundError(f"No CSV files found in {RAW_DIR}. This notebook expects CICMalDroid CSV input files.")

print("CSV files selected for loading:")
for path in csv_paths:
    print("-", path.relative_to(PROJECT_ROOT))


This cell defines helper functions for robust CSV loading, column-name cleanup, metadata detection, and original identifier detection.

In [ ]:
def load_csv(path):
    try:
        frame = pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        frame = pd.read_csv(path, low_memory=False, encoding="latin-1")

    # If pandas sees one huge column, retry with delimiter inference.
    if frame.shape[1] == 1:
        try:
            frame = pd.read_csv(path, sep=None, engine="python")
        except UnicodeDecodeError:
            frame = pd.read_csv(path, sep=None, engine="python", encoding="latin-1")
    return frame


def clean_column_names(columns):
    cleaned = []
    seen = {}
    for column in columns:
        base = str(column).strip()
        count = seen.get(base, 0)
        cleaned.append(base if count == 0 else f"{base}__duplicate_{count}")
        seen[base] = count + 1
    return cleaned


def normalize_name(name):
    return re.sub(r"[^a-z0-9]+", "", str(name).lower())


def find_first_column(columns, candidates):
    normalized = {normalize_name(column): column for column in columns}
    for candidate in candidates:
        key = normalize_name(candidate)
        if key in normalized:
            return normalized[key]
    for column in columns:
        col_key = normalize_name(column)
        if any(normalize_name(candidate) in col_key for candidate in candidates):
            return column
    return None


def clean_identifier_value(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except TypeError:
        pass
    text = str(value).strip()
    return text


IDENTIFIER_CANDIDATES = [
    "original_id", "dataset_key", "key", "sample_key", "json_filename", "json_file",
    "sha256", "sha1", "md5", "hash", "apk_hash", "apk", "apk_name", "apk_filename",
    "file", "filename", "file_name", "path", "file_path", "app", "app_name",
    "package", "package_name", "pkg_name", "name", "raw_identifier",
]
ORIGINAL_ID_CANDIDATES = [
    "original_id", "dataset_key", "key", "sample_key", "json_filename", "json_file",
    "apk_name", "apk_filename", "apk", "filename", "file_name", "file_path", "path",
    "sha256", "sha1", "md5", "hash", "apk_hash", "raw_identifier", "package_name",
]
ORDER_CANDIDATES = [
    "filename", "file_name", "file_path", "path", "apk_name", "apk", "file", "name",
]
LABEL_CANDIDATES = [
    "true_label", "label", "class", "classification", "malware", "is_malware",
    "benign_malware", "type", "verdict",
]
CATEGORY_CANDIDATES = [
    "category", "malware_category", "malware_type", "class", "classification", "type",
]
FAMILY_CANDIDATES = [
    "family", "malware_family", "family_name", "malware_name", "variant", "subclass",
]


This cell loads the raw CSV rows, adds source trace columns, and combines multiple raw files into a single master dataframe when needed.

In [ ]:
# Load all raw CSV rows first, without cleaning or inspecting the content.
loaded_frames = []
for path in csv_paths:
    frame = load_csv(path)
    frame.columns = clean_column_names(frame.columns)
    frame["source_file"] = str(path.relative_to(PROJECT_ROOT))
    frame["source_row_order"] = range(len(frame))
    loaded_frames.append({"path": path, "frame": frame})

if len(loaded_frames) == 1:
    master_df = loaded_frames[0]["frame"].copy()
else:
    normalized_column_sets = [
        set(map(normalize_name, item["frame"].columns)) - {"sourcefile", "sourceroworder"}
        for item in loaded_frames
    ]
    same_schema = all(columns == normalized_column_sets[0] for columns in normalized_column_sets)
    id_columns = [find_first_column(item["frame"].columns, IDENTIFIER_CANDIDATES) for item in loaded_frames]

    if same_schema:
        master_df = pd.concat([item["frame"] for item in loaded_frames], ignore_index=True, sort=False)
    elif all(id_columns):
        merge_frames = []
        for item, id_column in zip(loaded_frames, id_columns):
            frame = item["frame"].copy()
            prefix = re.sub(r"[^A-Za-z0-9]+", "_", item["path"].stem).strip("_").lower()
            frame["raw_identifier"] = frame[id_column].astype("string").fillna("unknown")
            rename_map = {
                column: f"{prefix}__{column}"
                for column in frame.columns
                if column not in {id_column, "raw_identifier"}
            }
            frame = frame.rename(columns=rename_map).drop(columns=[id_column])
            if frame["raw_identifier"].duplicated().any():
                frame = frame.drop_duplicates(subset=["raw_identifier"], keep="first")
            merge_frames.append(frame)
        master_df = reduce(lambda left, right: pd.merge(left, right, on="raw_identifier", how="outer"), merge_frames)
        source_cols = [column for column in master_df.columns if column.endswith("__source_file")]
        if source_cols:
            master_df["source_file"] = master_df[source_cols].apply(
                lambda row: "; ".join(str(value) for value in row.dropna().unique()), axis=1
            )
    else:
        master_df = pd.concat([item["frame"] for item in loaded_frames], ignore_index=True, sort=False)

print("Raw CSV files loaded:", len(loaded_frames))
print("Master dataframe rows before ordering:", len(master_df))


This cell builds the deterministic master ordering, creates `sample_id` and `original_id`, then writes the two contiguous raw shard CSV files.

In [ ]:
# Build deterministic ordering before any cleaning or prompt generation.
order_search_columns = [
    column for column in master_df.columns
    if not normalize_name(column).endswith("sourcefile") and normalize_name(column) != "sourceroworder"
]
order_column = find_first_column(order_search_columns, ORDER_CANDIDATES)
sort_columns = []
temporary_sort_columns = []

if order_column:
    master_df["__sort_original_path"] = master_df[order_column].astype("string").fillna("")
    sort_columns.append("__sort_original_path")
    temporary_sort_columns.append("__sort_original_path")
    print("Primary ordering column:", order_column)
else:
    print("Primary ordering column: none detected; using stable source file and row order.")

if "source_file" in master_df.columns:
    master_df["__sort_source_file"] = master_df["source_file"].astype("string").fillna("")
else:
    master_df["__sort_source_file"] = ""
sort_columns.append("__sort_source_file")
temporary_sort_columns.append("__sort_source_file")

if "source_row_order" in master_df.columns:
    master_df["__sort_row_order"] = pd.to_numeric(master_df["source_row_order"], errors="coerce").fillna(0).astype(int)
else:
    master_df["__sort_row_order"] = range(len(master_df))
sort_columns.append("__sort_row_order")
temporary_sort_columns.append("__sort_row_order")

master_df = master_df.sort_values(sort_columns, kind="mergesort").drop(columns=temporary_sort_columns).reset_index(drop=True)

# Preserve any existing columns before creating the canonical pipeline columns.
if "sample_id" in master_df.columns:
    master_df = master_df.rename(columns={"sample_id": "original_sample_id"})
if "source_dataset" in master_df.columns:
    master_df = master_df.rename(columns={"source_dataset": "original_source_dataset"})
if "original_id" in master_df.columns:
    master_df = master_df.rename(columns={"original_id": "raw_original_id"})

# original_id is the final merge key. It is built before sharding and never dropped.
original_id_source_col = None
if "raw_original_id" in master_df.columns:
    original_id_source_col = "raw_original_id"
elif "raw_identifier" in master_df.columns:
    original_id_source_col = "raw_identifier"
else:
    original_id_source_col = find_first_column(order_search_columns, ORIGINAL_ID_CANDIDATES)

fallback_source = (
    master_df["source_file"].astype("string").fillna("unknown_source")
    if "source_file" in master_df.columns
    else pd.Series(["unknown_source"] * len(master_df), index=master_df.index)
)
if "source_row_order" in master_df.columns:
    row_order_fallback = pd.Series(master_df.index, index=master_df.index)
    fallback_row = pd.to_numeric(master_df["source_row_order"], errors="coerce").fillna(row_order_fallback).astype(int).astype(str)
else:
    fallback_row = pd.Series(master_df.index.astype(str), index=master_df.index)
fallback_ids = fallback_source.astype(str) + "::row_" + fallback_row.astype(str)

if original_id_source_col and original_id_source_col in master_df.columns:
    base_original_ids = master_df[original_id_source_col].map(clean_identifier_value)
    base_original_ids = base_original_ids.where(base_original_ids != "", fallback_ids)
else:
    base_original_ids = fallback_ids
    original_id_source_col = "source_file + source_row_order fallback"

duplicate_base_count = int(base_original_ids.duplicated().sum())
occurrence = base_original_ids.groupby(base_original_ids).cumcount()
original_ids = base_original_ids.where(occurrence == 0, base_original_ids + "::dup_" + occurrence.astype(str))

master_df.insert(0, "sample_id", range(len(master_df)))
master_df.insert(1, "original_id", original_ids.astype(str))
master_df.insert(2, "source_dataset", SOURCE_DATASET)

print("Original ID source:", original_id_source_col)
print("Duplicate base original IDs before deterministic suffixing:", duplicate_base_count)
print("Unique original_id values:", master_df["original_id"].nunique())

midpoint = len(master_df) // 2
raw_shards = {
    1: master_df.iloc[:midpoint].copy().reset_index(drop=True),
    2: master_df.iloc[midpoint:].copy().reset_index(drop=True),
}

for shard_id, shard_df in raw_shards.items():
    shard_path = RAW_SHARD_PATHS[shard_id]
    shard_df.to_csv(shard_path, index=False)
    if shard_df.empty:
        sample_range = "empty"
        original_preview = "empty"
    else:
        sample_range = f"{int(shard_df['sample_id'].min())}..{int(shard_df['sample_id'].max())}"
        original_preview = f"{shard_df['original_id'].iloc[0]} -> {shard_df['original_id'].iloc[-1]}"
    print(f"Saved raw_shard{shard_id}: {len(shard_df)} rows, sample_id range {sample_range}, original_id range {original_preview}, path {shard_path}")

print("Split midpoint:", midpoint)
print("Total rows after split:", sum(len(frame) for frame in raw_shards.values()))


This cell optionally deletes only the unassigned intermediate raw shard while leaving `data/raw/` untouched.

In [ ]:
# Keep only the assigned intermediate shard when requested.
# This never deletes anything in data/raw and never deletes ZIP files or extracted raw folders.
opposite_shard_id = 2 if ASSIGNED_SHARD_ID == 1 else 1
opposite_path = RAW_SHARD_PATHS[opposite_shard_id]

if DELETE_UNASSIGNED_SHARD:
    if opposite_path.exists():
        opposite_path.unlink()
        print(f"Kept assigned shard {ASSIGNED_SHARD_ID}: {ASSIGNED_RAW_SHARD_PATH}")
        print(f"Removed unassigned intermediate shard {opposite_shard_id}: {opposite_path}")
    else:
        print(f"Kept assigned shard {ASSIGNED_SHARD_ID}: {ASSIGNED_RAW_SHARD_PATH}")
        print(f"Unassigned intermediate shard {opposite_shard_id} was already absent: {opposite_path}")
else:
    print("DELETE_UNASSIGNED_SHARD is False; keeping both raw shard CSV files.")


This cell loads the assigned raw shard, verifies that `original_id` is present, and displays the shard shape and identifier preview.

In [ ]:
# From this point onward, inspect and clean only the assigned shard.
if not ASSIGNED_RAW_SHARD_PATH.exists():
    raise FileNotFoundError(f"Assigned shard is missing: {ASSIGNED_RAW_SHARD_PATH}")

assigned_df = pd.read_csv(ASSIGNED_RAW_SHARD_PATH, low_memory=False)
if "original_id" not in assigned_df.columns:
    raise KeyError("Assigned raw shard must contain original_id. Re-run notebook 01 from the raw CICMalDroid files.")

print("Assigned raw shard shape:", assigned_df.shape)
print("Assigned raw shard columns:")
print(assigned_df.columns.tolist())
print("Assigned sample_id range:", int(assigned_df["sample_id"].min()), "to", int(assigned_df["sample_id"].max()))
print("Assigned original_id preview:", assigned_df["original_id"].head(3).tolist())
display(assigned_df.head())


This cell inspects missing values, inferred label/category/family columns, class distributions, and duplicate counts for the assigned shard.

In [ ]:
missing_summary = (
    assigned_df.isna().sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_percent=lambda data: (data["missing_count"] / len(assigned_df) * 100).round(2))
    .sort_values(["missing_count", "missing_percent"], ascending=False)
)
print("Missing values by column (top 50):")
display(missing_summary.head(50))

label_col = find_first_column(assigned_df.columns, LABEL_CANDIDATES)
category_col = find_first_column(assigned_df.columns, CATEGORY_CANDIDATES)
family_col = find_first_column(assigned_df.columns, FAMILY_CANDIDATES)
identifier_col = "raw_identifier" if "raw_identifier" in assigned_df.columns else find_first_column(assigned_df.columns, IDENTIFIER_CANDIDATES)

print("Detected identifier column:", identifier_col)
print("Detected label column:", label_col)
print("Detected category column:", category_col)
print("Detected family column:", family_col)

for title, column in [("Label", label_col), ("Category", category_col), ("Family", family_col)]:
    if column and column in assigned_df.columns:
        print(f"\n{title} distribution from column '{column}' (top 30):")
        print(assigned_df[column].value_counts(dropna=False).head(30))
    else:
        print(f"\n{title} distribution: no likely column detected")

print("\nDuplicate checks on assigned shard")
print("Full duplicate rows:", int(assigned_df.duplicated().sum()))
if identifier_col and identifier_col in assigned_df.columns:
    print(f"Duplicate identifiers in '{identifier_col}':", int(assigned_df[identifier_col].duplicated().sum()))


This cell identifies behavioral evidence columns while excluding metadata, labels, identifiers, and empty columns.

In [ ]:
metadata_norms = {normalize_name(value) for value in [
    identifier_col, label_col, category_col, family_col,
    "sample_id", "original_id", "source_dataset", "source_file", "source_row_order",
] if value}
metadata_keywords = [
    "label", "class", "category", "family", "verdict", "hash", "sha", "md5",
    "identifier", "package", "filename", "sourcefile",
]

evidence_columns = []
for column in assigned_df.columns:
    key = normalize_name(column)
    if key in metadata_norms:
        continue
    if any(keyword in key for keyword in metadata_keywords):
        continue
    if assigned_df[column].isna().all():
        continue
    evidence_columns.append(column)

text_evidence_columns = [column for column in evidence_columns if assigned_df[column].dtype == "object"]
numeric_evidence_columns = [column for column in evidence_columns if column not in text_evidence_columns]

print("Evidence columns detected:", len(evidence_columns))
print("Text/object evidence columns:", len(text_evidence_columns))
print("Numeric/boolean evidence columns:", len(numeric_evidence_columns))
print("First 80 evidence columns:")
print(evidence_columns[:80])


This cell performs light shard-local cleaning, preserves `original_id`, creates canonical metadata fields, and saves the assigned cleaned shard.

In [ ]:
# Light cleaning on the assigned shard only. Rows are preserved to keep shard ownership stable.
cleaned_df = assigned_df.copy()
if "original_id" not in cleaned_df.columns:
    raise KeyError("original_id must never be dropped. Re-run notebook 01 from the raw CICMalDroid files.")

empty_columns = [column for column in cleaned_df.columns if cleaned_df[column].isna().all() and column != "original_id"]
if empty_columns:
    print("Dropping fully empty columns from assigned shard:", empty_columns)
    cleaned_df = cleaned_df.drop(columns=empty_columns)
else:
    print("No fully empty columns to drop.")


def value_or_unknown(series):
    if series is None:
        return pd.Series(["unknown"] * len(cleaned_df), index=cleaned_df.index, dtype="string")
    return series.astype("string").fillna("unknown").replace("", "unknown")


label_col = find_first_column(cleaned_df.columns, LABEL_CANDIDATES)
category_col = find_first_column(cleaned_df.columns, CATEGORY_CANDIDATES)
family_col = find_first_column(cleaned_df.columns, FAMILY_CANDIDATES)
identifier_col = "raw_identifier" if "raw_identifier" in cleaned_df.columns else find_first_column(cleaned_df.columns, IDENTIFIER_CANDIDATES)
true_label_source = label_col or category_col or family_col

if "true_label" not in cleaned_df.columns:
    cleaned_df["true_label"] = value_or_unknown(cleaned_df[true_label_source]) if true_label_source else "unknown"
if "category" not in cleaned_df.columns:
    cleaned_df["category"] = value_or_unknown(cleaned_df[category_col]) if category_col else cleaned_df["true_label"]
if "family" not in cleaned_df.columns:
    cleaned_df["family"] = value_or_unknown(cleaned_df[family_col]) if family_col else "unknown"
if identifier_col and identifier_col in cleaned_df.columns and "raw_identifier" not in cleaned_df.columns:
    cleaned_df["raw_identifier"] = value_or_unknown(cleaned_df[identifier_col])
elif "raw_identifier" not in cleaned_df.columns:
    cleaned_df["raw_identifier"] = "unknown"

cleaned_df.to_csv(CLEANED_PATH, index=False)
print("Saved assigned cleaned CICMalDroid shard:", CLEANED_PATH)
print("Rows preserved:", len(cleaned_df))
print("Unique original_id values:", cleaned_df["original_id"].nunique())
print("\nFinal true_label distribution (top 30):")
print(cleaned_df["true_label"].value_counts(dropna=False).head(30))
display(cleaned_df.head())
